# 06 - Metaclassifiers

This notebook evaluates representative metaclassifier strategies studied in
the course.

The objective is to determine whether combining multiple classifiers improves
the probabilistic classification performance obtained in the previous PA2
experiments.

The following metaclassifier strategies are evaluated:

- Bagging.
- AdaBoost.
- Stacking.

The complete set of 30 valid predictors is used in this experiment so that the
effect of the metaclassifier itself can be analysed independently from feature
selection.

In [25]:
import pandas as pd
import numpy as np

from sklearn.base import clone

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler
)

from sklearn.pipeline import Pipeline

from sklearn.model_selection import StratifiedKFold

from sklearn.linear_model import LogisticRegression

from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis
)

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    BaggingClassifier,
    AdaBoostClassifier,
    StackingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 1. Dataset Preparation

The same PA2 dataset preparation is retained.

The identifier and constant attributes are removed and `Attrition` is encoded
as a binary target.

No feature-selection strategy is applied in this notebook.

In [26]:
DATA_PATH = "../../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv"

df = pd.read_csv(DATA_PATH)

columns_to_remove = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

df_model = df.drop(
    columns=columns_to_remove
)

X = df_model.drop(
    columns=["Attrition"]
)

y = df_model["Attrition"].map({
    "No": 0,
    "Yes": 1
})

print("Predictor matrix shape:", X.shape)
print("Target shape:", y.shape)

Predictor matrix shape: (1470, 30)
Target shape: (1470,)


In [27]:
ordinal_features = [
    "Education",
    "EnvironmentSatisfaction",
    "JobInvolvement",
    "JobLevel",
    "JobSatisfaction",
    "PerformanceRating",
    "RelationshipSatisfaction",
    "StockOptionLevel",
    "WorkLifeBalance"
]

numerical_features = (
    X.select_dtypes(include=np.number)
    .columns
    .tolist()
)

categorical_features = (
    X.select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

continuous_features = [
    feature
    for feature in numerical_features
    if feature not in ordinal_features
]

scaled_numeric_features = (
    continuous_features
    + ordinal_features
)

## 2. Numerical Representation

The metaclassifiers used in this experiment require numerical input.

The same preprocessing principle used for Logistic Regression is retained:

- continuous and ordinal predictors are standardized;
- nominal predictors are one-hot encoded;
- one reference category is dropped for each nominal attribute.

All preprocessing is fitted exclusively on the training data of each
cross-validation fold.

In [28]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            scaled_numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

## 3. Metaclassifier Design

Three representative ensemble strategies are evaluated.

### Bagging

Bagging trains multiple classifiers on bootstrap samples of the training data
and combines their predictions.

A Decision Tree is used as the base estimator because trees are naturally
suited to bootstrap aggregation.

### AdaBoost

AdaBoost sequentially trains weak learners while increasing the importance of
observations that were previously misclassified.

A shallow Decision Tree is used as the weak learner.

### Stacking

Stacking combines the predictions of multiple base classifiers using a
second-level classifier.

The base learners are:

- Logistic Regression.
- Linear Discriminant Analysis.
- Quadratic Discriminant Analysis.

Logistic Regression is used as the final estimator.

In [29]:
bagging_classifier = BaggingClassifier(
    estimator=DecisionTreeClassifier(
        random_state=42
    ),
    n_estimators=100,
    random_state=42
)

adaboost_classifier = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(
        max_depth=1,
        random_state=42
    ),
    n_estimators=100,
    random_state=42
)

In [30]:
stacking_classifier = StackingClassifier(
    estimators=[
        (
            "logistic",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        ),
        (
            "lda",
            LinearDiscriminantAnalysis()
        )
    ],
    final_estimator=LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    stack_method="predict_proba",
    cv=5,
    n_jobs=1
)

## 4. Complete Ensemble Pipelines

Each metaclassifier is placed after the preprocessing stage so that all
transformations are fitted only from the training portion of each outer
cross-validation fold.

In [31]:
bagging_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            clone(preprocessor)
        ),
        (
            "classifier",
            bagging_classifier
        )
    ]
)

adaboost_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            clone(preprocessor)
        ),
        (
            "classifier",
            adaboost_classifier
        )
    ]
)

stacking_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            clone(preprocessor)
        ),
        (
            "classifier",
            stacking_classifier
        )
    ]
)

## 5. Evaluation Strategy

The metaclassifiers are evaluated using the same stratified 10-fold
cross-validation used in the previous PA2 experiments.

This preserves comparability with the probabilistic baseline and feature-
selection experiments.

In [32]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [33]:
def multiclass_brier_score(
    y_true,
    probabilities,
    classes
):

    y_true = np.asarray(
        y_true
    )

    indicator_matrix = np.zeros_like(
        probabilities,
        dtype=float
    )

    for index, cls in enumerate(classes):

        indicator_matrix[:, index] = (
            y_true == cls
        ).astype(float)

    squared_error = (
        probabilities
        - indicator_matrix
    ) ** 2

    return np.mean(
        squared_error.sum(axis=1)
    )

In [34]:
def evaluate_metaclassifier(
    name,
    estimator,
    X,
    y,
    cv
):

    fold_results = []

    for fold, (
        train_index,
        test_index
    ) in enumerate(
        cv.split(X, y),
        start=1
    ):

        X_train = (
            X.iloc[
                train_index
            ].copy()
        )

        X_test = (
            X.iloc[
                test_index
            ].copy()
        )

        y_train = (
            y.iloc[
                train_index
            ].copy()
        )

        y_test = (
            y.iloc[
                test_index
            ].copy()
        )

        model = clone(
            estimator
        )

        model.fit(
            X_train,
            y_train
        )

        y_pred = model.predict(
            X_test
        )

        probabilities = model.predict_proba(
            X_test
        )

        classes = model.classes_

        positive_index = np.where(
            classes == 1
        )[0][0]

        positive_probability = (
            probabilities[
                :,
                positive_index
            ]
        )

        fold_results.append({
            "Model": name,
            "Fold": fold,

            "Accuracy":
                accuracy_score(
                    y_test,
                    y_pred
                ),

            "Precision":
                precision_score(
                    y_test,
                    y_pred,
                    zero_division=0
                ),

            "Recall":
                recall_score(
                    y_test,
                    y_pred
                ),

            "Specificity":
                recall_score(
                    y_test,
                    y_pred,
                    pos_label=0
                ),

            "F1":
                f1_score(
                    y_test,
                    y_pred,
                    zero_division=0
                ),

            "AUC":
                roc_auc_score(
                    y_test,
                    positive_probability
                ),

            "Brier":
                multiclass_brier_score(
                    y_test,
                    probabilities,
                    classes
                )
        })

    return pd.DataFrame(
        fold_results
    )

## 6. Metaclassifier Experiment

Bagging, AdaBoost, and Stacking are evaluated using the same outer
cross-validation partitions.

The experiment measures whether ensemble strategies improve classification or
probability quality relative to the individual probabilistic classifiers.

In [35]:
metaclassifiers = {
    "Bagging": bagging_pipeline,
    "AdaBoost": adaboost_pipeline,
    "Stacking": stacking_pipeline
}

all_meta_results = []

for name, estimator in metaclassifiers.items():

    print(
        f"Evaluating {name}..."
    )

    result = evaluate_metaclassifier(
        name=name,
        estimator=estimator,
        X=X,
        y=y,
        cv=cv
    )

    all_meta_results.append(
        result
    )

print(
    "\nMetaclassifier evaluation completed."
)

Evaluating Bagging...
Evaluating AdaBoost...
Evaluating Stacking...

Metaclassifier evaluation completed.


In [36]:
metaclassifier_fold_results = (
    pd.concat(
        all_meta_results,
        ignore_index=True
    )
)

display(
    metaclassifier_fold_results
    .round(4)
)

,Model,Fold,Accuracy,Precision,Recall,Specificity,F1,AUC,Brier
0,Bagging,1,0.8435,0.5714,0.1667,0.9756,0.2581,0.7710,0.2354
1,Bagging,2,0.8435,0.6000,0.1250,0.9837,0.2069,0.7551,0.2298
2,Bagging,3,0.8367,0.5000,0.2083,0.9593,0.2941,0.7764,0.2327
3,Bagging,4,0.8571,0.6154,0.3333,0.9593,0.4324,0.8089,0.2139
4,Bagging,5,0.8707,0.7778,0.2917,0.9837,0.4242,0.8039,0.1974
5,Bagging,6,0.8367,0.5000,0.2083,0.9593,0.2941,0.8362,0.2058
6,Bagging,7,0.8776,0.8000,0.3333,0.9837,0.4706,0.8538,0.1906
7,Bagging,8,0.8639,0.8000,0.1739,0.9919,0.2857,0.7845,0.2019
8,Bagging,9,0.8776,0.7778,0.3043,0.9839,0.4375,0.8087,0.2039
9,Bagging,10,0.8435,0.5000,0.2174,0.9597,0.3030,0.7319,0.2468


## 7. Cross-Validation Summary

Mean and standard deviation are calculated for every metric across the ten
validation folds.

In [37]:
metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "AUC",
    "Brier"
]

summary_rows = []

for model_name in (
    metaclassifier_fold_results[
        "Model"
    ].unique()
):

    model_data = (
        metaclassifier_fold_results[
            metaclassifier_fold_results[
                "Model"
            ] == model_name
        ]
    )

    row = {
        "Model": model_name
    }

    for metric in metrics:

        row[f"{metric}_mean"] = (
            model_data[
                metric
            ].mean()
        )

        row[f"{metric}_std"] = (
            model_data[
                metric
            ].std()
        )

    summary_rows.append(
        row
    )

metaclassifier_results = pd.DataFrame(
    summary_rows
)

display(
    metaclassifier_results
    .round(4)
)

,Model,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,Specificity_mean,Specificity_std,F1_mean,F1_std,AUC_mean,AUC_std,Brier_mean,Brier_std
0,Bagging,0.8551,0.0164,0.6442,0.1311,0.2362,0.0742,0.9740,0.0131,0.3407,0.0914,0.7931,0.0368,0.2158,0.0190
1,AdaBoost,0.8762,0.0231,0.7758,0.1706,0.3380,0.0738,0.9797,0.0176,0.4676,0.0958,0.8314,0.0518,0.3773,0.0083
2,Stacking,0.8830,0.0200,0.7472,0.1235,0.4259,0.0672,0.9708,0.0171,0.5394,0.0768,0.8365,0.0409,0.1846,0.0280


In [38]:
metaclassifier_summary = (
    metaclassifier_results[
        [
            "Model",
            "Accuracy_mean",
            "Precision_mean",
            "Recall_mean",
            "Specificity_mean",
            "F1_mean",
            "AUC_mean",
            "Brier_mean"
        ]
    ].copy()
)

metaclassifier_summary.columns = [
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "AUC",
    "Brier"
]

metaclassifier_summary = (
    metaclassifier_summary
    .sort_values(
        "F1",
        ascending=False
    )
)

display(
    metaclassifier_summary
    .round(4)
)

,Model,Accuracy,Precision,Recall,Specificity,F1,AUC,Brier
2,Stacking,0.8830,0.7472,0.4259,0.9708,0.5394,0.8365,0.1846
1,AdaBoost,0.8762,0.7758,0.3380,0.9797,0.4676,0.8314,0.3773
0,Bagging,0.8551,0.6442,0.2362,0.9740,0.3407,0.7931,0.2158


## 8. Comparison with Individual Probabilistic Classifiers

The metaclassifiers are compared with the individual probabilistic baseline.

This shows whether combining classifiers provides an advantage over the best
single-model configuration.

In [39]:
baseline_summary = pd.read_csv(
    "../../results/pa2/baseline_summary.csv"
)

individual_comparison = pd.concat(
    [
        baseline_summary.assign(
            Strategy="Individual"
        ),
        metaclassifier_summary.assign(
            Strategy="Metaclassifier"
        )
    ],
    ignore_index=True
)

display(
    individual_comparison[
        [
            "Strategy",
            "Model",
            "F1",
            "Recall",
            "AUC",
            "Brier"
        ]
    ]
    .sort_values(
        "F1",
        ascending=False
    )
    .round(4)
)

,Strategy,Model,F1,Recall,AUC,Brier
4,Metaclassifier,Stacking,0.5394,0.4259,0.8365,0.1846
0,Individual,Logistic Regression,0.5373,0.4259,0.8375,0.1857
1,Individual,Mixed Naive Bayes,0.5083,0.6453,0.7851,0.2902
5,Metaclassifier,AdaBoost,0.4676,0.3380,0.8314,0.3773
2,Individual,LDA,0.4256,0.3168,0.8146,0.2036
3,Individual,QDA,0.3946,0.3755,0.7211,0.2786
6,Metaclassifier,Bagging,0.3407,0.2362,0.7931,0.2158


## 9. Comparison with Previous PA2 Experiments

The metaclassifier results are also compared with the feature-selection
experiments performed previously.

In [40]:
univariate_summary = pd.read_csv(
    "../../results/pa2/univariate_summary.csv"
)

multivariate_summary = pd.read_csv(
    "../../results/pa2/multivariate_summary.csv"
)

wrapper_summary = pd.read_csv(
    "../../results/pa2/wrapper_summary.csv"
)

all_results = pd.concat(
    [
        baseline_summary.assign(
            Strategy="All Features"
        ),

        univariate_summary.assign(
            Strategy="Univariate"
        ),

        multivariate_summary.assign(
            Strategy="Multivariate"
        ),

        wrapper_summary.assign(
            Strategy="Wrapper"
        ),

        metaclassifier_summary.assign(
            Strategy="Metaclassifier"
        )
    ],
    ignore_index=True
)

display(
    all_results[
        [
            "Strategy",
            "Model",
            "F1",
            "Recall",
            "AUC",
            "Brier"
        ]
    ]
    .sort_values(
        "F1",
        ascending=False
    )
    .round(4)
)

,Strategy,Model,F1,Recall,AUC,Brier
16,Metaclassifier,Stacking,0.5394,0.4259,0.8365,0.1846
0,All Features,Logistic Regression,0.5373,0.4259,0.8375,0.1857
1,All Features,Mixed Naive Bayes,0.5083,0.6453,0.7851,0.2902
12,Wrapper,Logistic Regression,0.4696,0.3551,0.8206,0.1976
17,Metaclassifier,AdaBoost,0.4676,0.3380,0.8314,0.3773
4,Univariate,Mixed Naive Bayes,0.4600,0.6033,0.7633,0.3190
13,Wrapper,Mixed Naive Bayes,0.4358,0.4087,0.7701,0.2400
2,All Features,LDA,0.4256,0.3168,0.8146,0.2036
5,Univariate,QDA,0.4240,0.3924,0.7302,0.2628
8,Multivariate,Mixed Naive Bayes,0.3962,0.3531,0.7616,0.2386


## 10. Best Overall Configurations

The best configurations according to F1, Recall, AUC, and Brier score are
identified across all experiments completed so far.

In [41]:
best_f1 = all_results.loc[
    all_results["F1"].idxmax()
]

best_recall = all_results.loc[
    all_results["Recall"].idxmax()
]

best_auc = all_results.loc[
    all_results["AUC"].idxmax()
]

best_brier = all_results.loc[
    all_results["Brier"].idxmin()
]

print("Best F1:")
display(
    best_f1.to_frame().T[
        [
            "Strategy",
            "Model",
            "F1"
        ]
    ]
)

print("\nBest Recall:")
display(
    best_recall.to_frame().T[
        [
            "Strategy",
            "Model",
            "Recall"
        ]
    ]
)

print("\nBest AUC:")
display(
    best_auc.to_frame().T[
        [
            "Strategy",
            "Model",
            "AUC"
        ]
    ]
)

print("\nBest Brier:")
display(
    best_brier.to_frame().T[
        [
            "Strategy",
            "Model",
            "Brier"
        ]
    ]
)

Best F1:


,Strategy,Model,F1
16,Metaclassifier,Stacking,0.539435



Best Recall:


,Strategy,Model,Recall
1,All Features,Mixed Naive Bayes,0.64529



Best AUC:


,Strategy,Model,AUC
0,All Features,Logistic Regression,0.837485



Best Brier:


,Strategy,Model,Brier
16,Metaclassifier,Stacking,0.184553


## 11. Export Results

The metaclassifier fold-level and aggregated results are stored for the final
PA2 analysis.

In [42]:
metaclassifier_fold_results.to_csv(
    "../../results/pa2/metaclassifier_fold_results.csv",
    index=False
)

metaclassifier_results.to_csv(
    "../../results/pa2/metaclassifier_results.csv",
    index=False
)

metaclassifier_summary.to_csv(
    "../../results/pa2/metaclassifier_summary.csv",
    index=False
)

print(
    "PA2 metaclassifier results saved successfully."
)

PA2 metaclassifier results saved successfully.


## 12. Main Observations

Stacking obtains the strongest metaclassifier performance, with a mean
F1-measure of 0.5394, AUC of 0.8365, and Brier score of 0.1846.

Its F1-measure is slightly higher than the best individual probabilistic model,
Logistic Regression with all features, which obtains F1 = 0.5373. Stacking
also achieves a slightly better Brier score than Logistic Regression
(0.1846 versus 0.1857).

However, the AUC of Stacking (0.8365) is marginally lower than the AUC of the
full-feature Logistic Regression baseline (0.8375), and both configurations
obtain the same Recall (0.4259).

AdaBoost obtains intermediate performance, with F1 = 0.4676 and AUC = 0.8314.
Its Precision and Specificity are high, but Recall remains relatively low
(0.3380). Its Brier score (0.3773) is substantially worse than the other
competitive configurations.

Bagging provides the weakest metaclassifier results, with F1 = 0.3407 and
Recall = 0.2362, despite maintaining high Specificity.

Across all PA2 experiments completed so far, Stacking obtains the highest F1
and the lowest Brier score, while the full-feature Logistic Regression
maintains the highest AUC. Mixed Naive Bayes with all features retains the
highest Recall.

Overall, Stacking provides a small improvement in F1-measure and probability
quality over the strongest individual classifier, although the gain is modest
and does not improve AUC or Recall.